In [8]:
from dotenv import load_dotenv
import os
load_dotenv(override=True)

True

In [9]:
from typing import Literal
from langchain.messages import HumanMessage
from langchain_anthropic import ChatAnthropic
from langchain.tools import tool
from langgraph.prebuilt import ToolNode
from langgraph.graph import StateGraph, MessagesState

@tool
def search(query: str):
    """Call to surf the web."""
    if "sf" in query.lower() or "san francisco" in query.lower():
        return "It's 60 degrees and foggy."
    return "It's 90 degrees and sunny."

tools = [search]
tool_node = ToolNode(tools)

model = ChatAnthropic(model="claude-haiku-4-5-20251001", temperature=0).bind_tools(tools)

def should_continue(state: MessagesState) -> Literal["tools", "__end__"]:
    messages = state['messages']
    last_message = messages[-1]
    if last_message.tool_calls:
        return "tools"
    return "__end__"

def call_model(state: MessagesState):
    messages = state['messages']
    # Invoking `model` will automatically infer the correct tracing context
    response = model.invoke(messages)
    return {"messages": [response]}

workflow = StateGraph(MessagesState)
workflow.add_node("agent", call_model)
workflow.add_node("tools", tool_node)
workflow.add_edge("__start__", "agent")
workflow.add_conditional_edges(
    "agent",
    should_continue,
)
workflow.add_edge("tools", 'agent')

app = workflow.compile()

final_state = app.invoke(
    {"messages": [HumanMessage(content="what is the weather in sf")]},
    config={"configurable": {"thread_id": 45}}
)

final_state["messages"][-1].content

"The current weather in San Francisco is **60°F and foggy**. This is pretty typical for SF, especially if it's morning or evening hours when the famous fog rolls in!"

In [16]:
## need to display the workflow visualization in the notebook
from IPython.display import display
display(workflow)